# Colab L4 — Area / Traffic / PPAC · 0.7.1rc2

Notebook16 menyediakan tiga mode independen:

| Mode | Engine yang dijalankan | PPAC |
| --- | --- | --- |
| Area Coverage | Area saja | Residential + Fasum; STD total belum lengkap |
| Traffic Counter | Traffic untuk video valid saja | Komponen Traffic; observasi kurang ditandai sementara, STD total belum lengkap |
| PPAC | Area → Traffic | STD total hanya bila sumber/observasi wajib lengkap |

Pilih runtime L4, jalankan setup sekali dan autentikasi Google, kemudian buka UI. Tidak perlu menjalankan Area atau membuka job Area lebih dahulu untuk memakai Traffic Counter maupun PPAC. Identitas dan parameter bisnis wajib jelas. Video harus lolos inspeksi dan tinjauan geometri; video yang tidak valid dilewati dengan alasan. Durasi yang tidak memenuhi policy300–306detik tetap boleh dicacah tetapi tidak otomatis menjadi sampel PPAC5menit. Mode PPAC tetap membutuhkan konfirmasi video/lokasi.

Untuk STD gabungan: weekday/weekend × pagi/siang/sore/malam wajib lengkap; jam operasional>15 memerlukan sesi tambahan. Hasil komponen tersedia dipertahankan bila STD total BLOCKED. Sesi hilang tidak dibuat sebagai sampel nol. Kebijakan bangunan alokasi+UNVERIFIED tetap, status UNVERIFIED tidak diubah menjadi terverifikasi.

HOME/RESULT, peta Folium/Esri, upload8video, GCS gs://aretfak-location-intelligence dan BQ locdeproject.location_intelligence tetap. Dataset/tabel/IAM tidak dibuat otomatis. Semua perhitungan memakai input dalam job yang sama; notebook tidak menggabungkan job lama secara diam-diam. Server dan tunnel ditutup saat cell server dihentikan. Kandidat pengujian; belum membuktikan kesiapan produksi.


In [ ]:
import csv,io,json,os,subprocess,sys,time,signal
from pathlib import Path
BASE=Path('/content/location-intelligence-colab');BASE.mkdir(parents=True,exist_ok=True)
try:
    output=subprocess.check_output(['nvidia-smi','--query-gpu=name,uuid,memory.total,driver_version','--format=csv,noheader,nounits'],text=True,timeout=30)
except (OSError,subprocess.SubprocessError):
    raise RuntimeError('GPU belum tersedia. Pilih Runtime → Change runtime type → L4 GPU lalu reconnect.') from None
GPU=[dict(zip(('name','uuid','memory_mib','driver'),(v.strip() for v in row))) for row in csv.reader(io.StringIO(output)) if len(row)==4]
import re
if not GPU or not all(re.search(r'\bL4\b',v['name']) for v in GPU):
    raise RuntimeError('GPU yang benar-benar dialokasikan bukan L4: '+str(GPU)+'. Pilih L4 pada pengaturan Colab.')
print('GPU_ALLOCATION='+json.dumps(GPU))
(BASE/'gpu-allocation.json').write_text(json.dumps(GPU,indent=2))


## Setup sekali di awal
Environment scientific dan core memakai lock terpisah. Transitive pip memakai hash; conda memakai URL build dan SHA256. Biner ffmpeg/osmium dipasang melalui apt jika belum tersedia; versi sistem tetap diperiksa. Jangan menjalankan ulang setup saat server/engine masih aktif.

In [ ]:
import base64,hashlib,zipfile,io
if 'SERVER' in globals() and SERVER.poll() is None:
    raise RuntimeError('Hentikan cell server dahulu sebelum menjalankan setup ulang.')
PAYLOAD=base64.b64decode('...')
assert hashlib.sha256(PAYLOAD).hexdigest()=='4694fec47d0b0bf9eddd7c34db06752c8cec901e7448dc85cf1b79b23851a9d5'
with zipfile.ZipFile(io.BytesIO(PAYLOAD)) as bundle:
    bundle.extractall(BASE)
WHEEL=BASE/'package'/'traffic_location_engines-0.7.1rc2-py3-none-any.whl'
assert hashlib.sha256(WHEEL.read_bytes()).hexdigest()=='d84fb558fa33b6a5a484caf5088c3e928c978fec70f3d74a2c2daad671ba26bf'
def run_logged(command,log_name,env=None):
    # Python-level streaming works in Colab even when inherited child FDs are hidden.
    # Keep the Colab kernel's inline backend out of application subprocesses.
    env={**(os.environ if env is None else env),'MPLBACKEND':'Agg'}
    import re
    log_path=BASE/(log_name+'.log')
    print('Menjalankan '+log_name+'; log: '+str(log_path),flush=True)
    with log_path.open('w') as logs:
        process=subprocess.Popen(command,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                 text=True,encoding='utf-8',errors='replace',bufsize=1,start_new_session=True)
        hiding_key=False
        try:
            for line in process.stdout:
                if '-----BEGIN' in line and 'PRIVATE KEY' in line:
                    hiding_key=True;line='[REDACTED PRIVATE KEY]\n'
                elif hiding_key:
                    if '-----END' in line and 'PRIVATE KEY' in line:hiding_key=False
                    continue
                line=re.sub(r'(?i)(bearer\s+)[A-Za-z0-9._~+/-]+',r'\1[REDACTED]',line)
                line=re.sub(r'(?i)(private_key|access_token|refresh_token|authorization)["\s:=]+[^\n]+',r'\1=[REDACTED]',line)
                logs.write(line);logs.flush();print(line,end='',flush=True)
            returncode=process.wait()
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGTERM)
                try:process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid,signal.SIGKILL);process.wait(timeout=5)
            raise
        finally:
            if process.stdout is not None:process.stdout.close()
    if returncode:
        status_path=BASE/('setup-status.json' if log_name=='setup' else 'colab-startup-failure.json')
        try:status=json.loads(status_path.read_text())
        except (OSError,ValueError):status={}
        with log_path.open('rb') as logs:
            logs.seek(max(0,log_path.stat().st_size-16384));tail=logs.read().decode('utf-8',errors='replace')
        tail='\n'.join(tail.splitlines()[-80:])
        raise RuntimeError(log_name+' gagal pada tahap '+str(status.get('stage','lihat log'))+
                           '; exit='+str(returncode)+'; '+str(log_path)+'\n'+tail)
    return returncode
run_logged([sys.executable,'-u',str(BASE/'bootstrap.py'),str(BASE),'--wheel',WHEEL.name],'setup')
"""Prepare the loader before exec: CUDA wheels are outside the system ld cache.

Standard library only; embedded in the notebook and existing-session recovery.
Does not install packages, load CUDA into the host kernel, or alter os.environ.
"""
def cuda_child_environment(science_python, environment):
    import json
    import subprocess
    from pathlib import Path
    env = dict(environment)
    # Ask the application interpreter, never the notebook's Python/version.
    paths = json.loads(subprocess.check_output(
        [str(science_python), '-I', '-c',
         'import json,sysconfig;print(json.dumps(sysconfig.get_paths()))'],
        env=env, text=True, timeout=30))
    site = Path(paths['purelib'])
    required = (site/'nvidia/cuda_nvrtc/lib/libnvrtc.so.12',
                site/'nvidia/cuda_runtime/lib/libcudart.so.12')
    missing = [str(p) for p in required if not p.is_file()]
    if missing:
        raise RuntimeError('CUDA wheel library belum terpasang: '+', '.join(missing)+
                           '. Jalankan ulang cell setup terkunci; jangan upgrade paket kernel Colab.')
    libraries = sorted(p for p in (site/'nvidia').glob('*/lib') if p.is_dir())
    if (site/'torch/lib').is_dir():
        libraries.append(site/'torch/lib')
    # Set this BEFORE Popen: changing LD_LIBRARY_PATH inside a running Python
    # process does not reliably change glibc's dlopen search path.
    inherited = [p for p in env.get('LD_LIBRARY_PATH', '').split(':') if p]
    env['LD_LIBRARY_PATH'] = ':'.join(dict.fromkeys([str(p) for p in libraries]+inherited))
    return env

SETUP=json.loads((BASE/'setup-ready.json').read_text())
SCIENCE=SETUP['science_python'];CORE=SETUP['core_python']
def child_env():
    env={**os.environ,'LI_RUNTIME_MODE':'bundled','LI_AUTH_MODE':'adc','LI_RUNTIME_PLATFORM':'colab','LI_COMPONENT_FLOW':'1',
         'LI_CORE_PYTHON':CORE,'LI_COLAB_GATE_ROOT':str(BASE/'area-verifications'),
         'GRADIO_ANALYTICS_ENABLED':'False','MPLBACKEND':'Agg','PROJ_NETWORK':'OFF','GOOGLE_CLOUD_PROJECT':'locdeproject'}
    for key in ('PROJ_LIB','PROJ_DATA','GDAL_DATA'):env.pop(key,None)
    return cuda_child_environment(SCIENCE,env)
def command(operation):
    return [SCIENCE,'-u','-m','traffic_location.integration.colab_runtime',operation,'--base',str(BASE),'--config-root',str(BASE/'config')]


## Periksa import dan CUDA
Pemeriksaan ini menjalankan tensor/kernel sintetis kecil pada CuPy, Torch, dan ONNX Runtime CUDA. Core diperiksa ulang dengan PYTHONPATH worker untuk menangkap pencampuran dependency sebelum job Area. Tidak menjalankan Area/Traffic atau model inference. Provider CUDA harus benar-benar bekerja; tidak ada fallback CPU tersembunyi.

In [ ]:
run_logged(command('verify'),'runtime-check',env=child_env())
# Notebook fragment: run after verify, before authentication or engine execution.
core_isolation_probe = r'''
import json,os,subprocess,sys,sysconfig
from pathlib import Path
base=Path(sys.argv[1]);core=sys.argv[2]
probe="""import json,platform,pyproj
from osgeo import gdal
import traffic_location.location.area._kernel.location_intelligent_li3_core
print(json.dumps({'python':platform.python_version(),'gdal':gdal.VersionInfo('RELEASE_NAME'),'pyproj':pyproj.__version__,'pyproj_file':pyproj.__file__,'gdal_file':gdal.__file__}))"""
# Exact PYTHONPATH shape set by the installed Area pipeline.
worker_env={**os.environ,'PYTHONPATH':sysconfig.get_path('purelib')+os.pathsep+os.environ.get('PYTHONPATH','')}
clean=json.loads(subprocess.check_output([core,'-c',probe],text=True,timeout=60))
worker=json.loads(subprocess.check_output([core,'-c',probe],env=worker_env,text=True,timeout=60))
assert clean==worker,('AREA_CORE_ISOLATION_FAILED',clean,worker)
core_root=(base/'runtime/core').resolve()
assert all(Path(worker[k]).resolve().is_relative_to(core_root) for k in ('pyproj_file','gdal_file')),worker
report={'status':'PASS','core':worker,'scope':'Core import and worker PYTHONPATH isolation; not Area execution.'}
(base/'core-runtime-isolation.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report))
'''
run_logged([SCIENCE,'-u','-c',core_isolation_probe,str(BASE),CORE],'core-isolation',env=child_env())


## Autentikasi Google dan pemeriksaan akses
Dialog ini untuk akun Google pemilik pengujian. ADC dipakai proses terisolasi dan worker. Pemeriksaan mencakup metadata sumber, schema/izin tabel hasil, izin object GCS, serta query kecil. Keberhasilan write/publish/reopen nyata tetap diuji lewat job Area; IAM check bukan transaksi write.
Akses cloud disiapkan sebelum UI; pemilihan mode menentukan engine yang dieksekusi.

In [ ]:
from google.colab import auth
auth.authenticate_user(project_id='locdeproject')
run_logged(command('preflight'),'cloud-preflight',env=child_env())
print('Akses awal diperiksa. Belum ada engine berjalan.')


## Jalankan UI — pilih Area Coverage, Traffic Counter atau PPAC
Pilih mode pada HOME dan Process. Mode Area dan Traffic menghitung komponen PPAC masing-masing. Mode PPAC menjalankan Area → Traffic → PPAC gabungan. Periksa scope, kualitas, video SKIPPED dan reason pada RESULT. Diagnosis berjalan pada antrean tersendiri.

**Biarkan cell server tetap berjalan. Stop/Interrupt menutup server DAN tunnel; URL publik menjadi tidak aktif.** Status proses dan HTTP lokal dicetak setiap30detik. HTTP lokal READY tidak membuktikan tunnel/browser maupun engine sehat. Hentikan setelah job selesai dan simpan diagnosis sebelum runtime Colab berakhir.

In [ ]:
"""Notebook-only process and loopback HTTP monitoring; no public tunnel probe."""
def server_snapshot(process, base, previous=None):
    import datetime,json,re,urllib.request,urllib.error
    from pathlib import Path
    base=Path(base);previous=previous or {}
    local=previous.get('local_url');public=previous.get('public_url')
    log=base/'server.log'
    if log.is_file():
        with log.open('rb') as stream:
            stream.seek(max(0,log.stat().st_size-65536))
            tail=stream.read().decode('utf-8',errors='replace')
        ports=re.findall(r'Running on local URL:\s+http://(?:0\.0\.0\.0|127\.0\.0\.1|localhost):(\d+)',tail)
        if ports and 0<int(ports[-1])<65536:local='http://127.0.0.1:'+ports[-1]
        urls=re.findall(r'(?:GRADIO_PUBLIC_URL=|Running on public URL:\s+)(https://[a-zA-Z0-9.-]+\.gradio\.live)\b',tail)
        if urls:public=urls[-1]
    exit_code=process.poll()
    http={'status':'NOT_CHECKED','url':local+'/config' if local else None}
    if exit_code is None and local:
        try:
            # Loopback only. Do not send local health probes through an HTTP proxy.
            class NoRedirect(urllib.request.HTTPRedirectHandler):
                def redirect_request(self,*args,**kwargs):return None
            opener=urllib.request.build_opener(urllib.request.ProxyHandler({}),NoRedirect())
            with opener.open(http['url'],timeout=2) as response:
                code=response.status;data=response.read(1024*1024+1)
            if len(data)>1024*1024:raise ValueError('CONFIG_RESPONSE_TOO_LARGE')
            config=json.loads(data)
            http.update(code=code,status='READY' if isinstance(config,dict) and isinstance(config.get('components'),list) else 'INVALID_RESPONSE')
        except urllib.error.HTTPError as error:http.update(status='HTTP_ERROR',code=error.code)
        except Exception as error:http.update(status='UNREACHABLE',error_type=type(error).__name__)
    status=('STOPPED' if exit_code is not None else
            'RUNNING_LOCAL_HTTP_OK' if http['status']=='READY' else
            'STARTING' if not local else 'RUNNING_LOCAL_HTTP_UNAVAILABLE')
    return {'schema_version':'colab_server_status_v1',
            'observed_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'status':status,'pid':process.pid,'exit_code':exit_code,'local_url':local,'public_url':public,
            'local_http':http,'public_tunnel':'NOT_PROBED',
            'scope':'Local HTTP readiness only; does not prove public tunnel/browser access or engine success.'}


def record_server_status(process,base,previous=None,reason=None):
    import json
    from pathlib import Path
    report=server_snapshot(process,base,previous)
    if reason:report['stop_reason']=reason
    if report['exit_code'] is not None and previous:
        report['last_running_observation']={k:previous.get(k) for k in
            ('observed_at','status','local_http','public_tunnel')}
    path=Path(base)/'server-status.json';temporary=path.with_suffix('.partial')
    temporary.write_text(json.dumps(report,indent=2));temporary.replace(path)
    print('GRADIO_STATUS='+json.dumps(report),flush=True)
    return report

if 'SERVER' in globals() and SERVER.poll() is None:
    raise RuntimeError('Server masih aktif; hentikan dahulu.')
log_path=BASE/'server.log';stream=log_path.open('wb')
SERVER=subprocess.Popen(command('serve'),env=child_env(),stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
offset=0;health=None;next_health=0;stop_reason='PROCESS_EXITED'
print('PERHATIAN: menghentikan/interrupt cell ini MENUTUP server Gradio dan tunnel. URL publik tidak dapat dipakai setelah server ditutup. Biarkan cell berjalan selama memakai UI.',flush=True)
try:
    while SERVER.poll() is None:
        with log_path.open('rb') as logs:
            logs.seek(offset);chunk=logs.read();offset=logs.tell()
        if chunk:print(chunk.decode('utf-8',errors='replace'),end='',flush=True)
        if time.monotonic()>=next_health:
            health=record_server_status(SERVER,BASE,health);next_health=time.monotonic()+30
            if health['status']=='RUNNING_LOCAL_HTTP_OK':
                print('HTTP lokal siap. Jika URL publik timeout, periksa tunnel/jaringan browser; akses publik belum diuji oleh probe lokal.',flush=True)
        time.sleep(1)
    with log_path.open('rb') as logs:
        logs.seek(offset);print(logs.read().decode('utf-8',errors='replace'),end='')
    if SERVER.returncode:raise RuntimeError('Server berhenti; lihat pesan terakhir dan server.log. Jangan lanjut ke Traffic sebelum Area terverifikasi.')
except KeyboardInterrupt:
    stop_reason='NOTEBOOK_INTERRUPT'
    print('Cell di-interrupt: server dan tunnel Gradio ditutup. Jalankan kembali cell ini lalu gunakan URL baru untuk membuka UI.',flush=True)
finally:
    if SERVER.poll() is None:
        os.killpg(SERVER.pid,signal.SIGINT)
        try:SERVER.wait(timeout=20)
        except subprocess.TimeoutExpired:
            os.killpg(SERVER.pid,signal.SIGTERM)
            try:SERVER.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(SERVER.pid,signal.SIGKILL);SERVER.wait(timeout=5)
    stream.close()
    health=record_server_status(SERVER,BASE,health,reason=stop_reason)
    print('Gradio BERHENTI. URL publik sesi ini tidak aktif. Jalankan kembali cell server untuk memperoleh URL aktif.',flush=True)


## Unduh diagnosis setelah server dihentikan
Tombol Siapkan paket diagnosis Area juga tersedia di Gradio. Cell berikut tetap bisa diunduh bila setup gagal sebelum Python aplikasi tersedia. ZIP memuat server.log, server-status.json, tahap gagal dan tail log terbatas; jika Area pernah berjalan, laporan run terakhir juga disertakan. Tidak memuat file credential/source/video. Unduhan ini tidak menjalankan ulang engine.

In [ ]:
# This cell also works when setup failed before SCIENCE was assigned.
import zipfile,re
archive=BASE/'colab-diagnostics.zip'
exported=False
if 'SCIENCE' in globals() and Path(SCIENCE).is_file():
    result=subprocess.run([SCIENCE,'-c',"from pathlib import Path\nimport sys,json,zipfile\nfrom traffic_location.integration.area_trace import latest_report\nfrom traffic_location.integration.failure_details import safe\nfrom traffic_location.integration.server_watchdog import observation\nbase=Path(sys.argv[1]);candidates=[(p.stat().st_mtime,p.parents[2]) for p in (base/'system/targets').glob('*/runs/*/job-input.json')]\ncandidates.extend((p.stat().st_mtime,p.parents[3]) for p in (base/'system/targets').glob('*/runs/*/area/area-trace.json'))\nreport=latest_report(max(candidates,key=lambda v:v[0])[1]) if candidates else {'status':'NO_ENGINE_RUN'}\ndef clean(v):\n    if isinstance(v,str):return safe(v)\n    if isinstance(v,dict):return {k:clean(x) for k,x in v.items()}\n    if isinstance(v,list):return [clean(x) for x in v]\n    return v\nwith zipfile.ZipFile(base/'colab-diagnostics.zip','w',zipfile.ZIP_DEFLATED) as z:\n    z.writestr('area-report.json',json.dumps(clean(report),indent=2))\n    z.writestr('server-observation.json',json.dumps(clean(observation(base,public=True)),indent=2))\n    for name in ('gpu-allocation.json','colab-runtime-verified.json','colab-preflight.json','colab-startup-failure.json','setup-status.json','core-runtime-isolation.json'):\n        path=base/name\n        if path.is_file() and path.stat().st_size<131072:z.writestr(name,json.dumps(clean(json.loads(path.read_text())),indent=2))\nprint('DIAGNOSTIC_ZIP='+str(base/'colab-diagnostics.zip'))\n",str(BASE)],env=child_env(),capture_output=True,text=True)
    exported=result.returncode==0
    if exported:print(result.stdout)
with zipfile.ZipFile(archive,'a' if exported else 'w',zipfile.ZIP_DEFLATED) as bundle:
    if not exported:bundle.writestr('status.txt','Setup/startup diagnosis only; Area report not available yet.')
    for name in ('setup-status.json','setup.log','runtime-check.log','cloud-preflight.log','core-isolation.log','core-runtime-isolation.json','server.log','server-status.json','server-lifecycle.json','server-watchdog.json','server-stacks.log'):
        path=BASE/name
        if not path.is_file() or name in bundle.namelist():continue
        with path.open('rb') as stream:
            stream.seek(max(0,path.stat().st_size-65536));text=stream.read().decode('utf-8',errors='replace')
        text=re.sub(r'-----BEGIN[^\n]*PRIVATE KEY-----.*?-----END[^\n]*PRIVATE KEY-----','[REDACTED_KEY]',text,flags=re.S)
        text=re.sub(r'(?i)(bearer\s+)[A-Za-z0-9._~+/-]+',r'\1[REDACTED]',text)
        text=re.sub(r'(?i)(private_key|access_token|refresh_token|authorization)["\s:=]+[^\n]+',r'\1=[REDACTED]',text)
        bundle.writestr(name,text)
from google.colab import files
files.download(str(archive))
